# Prefect lesson: worked solutions to the exercises

Run this notebook top to bottom (about 8 minutes). It runs the real `pipeline.py` from this folder, but in **temporary
folders** with a **temporary Prefect server** and **temporary MLflow stores**, so it changes nothing of yours: your `data/`,
`reports/`, `mlflow.db`, Prefect runs and registered models are not touched. Nothing goes over the network.
Each exercise shows the question, the code that answers it, and an **Answer** based on the numbers printed above it.
Participants: if you could not solve an exercise, or want to check your own answer, just run the cells.

In [ ]:
import os, sys, shutil, subprocess, types, tempfile, warnings, json, io, time, threading, logging, importlib.util, contextlib
from pathlib import Path

# Everything below lives in one temporary folder, including Prefect's own database.
TMP = Path(tempfile.mkdtemp(prefix="prefect_solutions_"))
os.environ["PREFECT_HOME"] = str(TMP / "prefect_home")
os.environ["PREFECT_LOGGING_LEVEL"] = "WARNING"                   # quiet: warnings only (blocked files, alerts)
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_LOGGING_LEVEL"] = "WARNING"
os.environ["DVC_NO_ANALYTICS"] = "1"
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]   # the pipeline calls `dvc`
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 70, "display.precision", 3)

# A throwaway Prefect server + database for the whole notebook (closed in the last cell).
harness = prefect_test_harness()
harness.__enter__()

HERE = Path.cwd()                                  # 06-prefect: the real lesson folder (we only read from it)
ROOT = HERE.parent


def fresh_pipeline(name):
    """Copy the REAL pipeline.py (and the helpers it imports) into a temp 'repo' and import it from there.

    pipeline.py creates data/, reports/, mlflow.db next to itself, so this keeps all of that in the temp folder.
    The code is unchanged; settings such as TRAIN_WINDOW are patched on the imported module.
    """
    top = TMP / name
    for folder, pattern in [("00-data", "*.py"), ("02-great-expectations", "*.py"),
                            ("03-mlflow", "*.py"), ("06-prefect", "pipeline.py")]:
        (top / folder).mkdir(parents=True)
        for f in (ROOT / folder).glob(pattern):
            shutil.copy(f, top / folder / f.name)
    gx_copy = top / "02-great-expectations" / "gx"
    shutil.copytree(ROOT / "02-great-expectations" / "gx", gx_copy, ignore=shutil.ignore_patterns("uncommitted"))
    with open(gx_copy / "great_expectations.yml", "a") as f:      # no progress bars in the notebook output
        f.write("\nprogress_bars:\n  globally: false\n  metric_calculations: false\n")
    subprocess.run(["git", "init", "-q"], cwd=top, check=True)    # `dvc init` wants a git repo
    spec = importlib.util.spec_from_file_location(f"pipeline_{name}", top / "06-prefect" / "pipeline.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    # hide dvc's progress spinner (the pipeline calls dvc through subprocess.run)
    mod.subprocess = types.SimpleNamespace(
        run=lambda *a, **k: subprocess.run(*a, **{**k, "stderr": subprocess.DEVNULL, "stdout": subprocess.DEVNULL}))
    return mod


def quiet(fn, *args, **kwargs):
    """Run a flow, hiding the table it prints at the end."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

print("temporary folder:", TMP)

### Setup: a pipeline with a 2024 history and a first champion

Exercises 1, 4 and 5 start from the same place: the `bootstrap` flow (12 months of 2024, DVC-versioned, first model
registered as `@champion`) plus a normal January 2025. About a minute.

In [ ]:
t0 = time.time()
pl = fresh_pipeline("base")
print("pipeline data lives in:", pl.HERE)
pl.bootstrap()
print(f"bootstrap done in {time.time() - t0:.0f}s; months in data/raw: {len(pl.accepted_months())}")

## Exercise 1

**Alert someone.** When a month is BLOCKED, the pipeline only logs it. Add a notification (Prefect *Automations* in the UI
can react to a flow's state, or call a webhook from the flow).

A blocked month is not a failed run: `monthly_pipeline` *returns* a row with `status = "BLOCKED by GX"`, so the flow ends
`Completed` and an Automation on "flow failed" would never fire. Two ways to handle it:

- an **Automation** in the UI that reacts to a flow-run state would need the pipeline to fail or add a tag/event on blocked months;
- a **flow hook** in code (below): `on_completion` receives the finished run, we read the returned row and notify when it is blocked.

The hook is attached with `with_options`, so `pipeline.py` itself is not edited. Here the "notification" is a log line plus a line
in an outbox file; a real one is a single call (shown in the comment).

In [ ]:
OUTBOX = TMP / "outbox.txt"                                   # stands in for Slack / e-mail / a webhook
alert_log = logging.getLogger("pipeline-alerts")


def notify_if_blocked(flow, flow_run, state):
    """Flow hook: runs when a monthly_pipeline run finishes. Notify only if the month was blocked."""
    row = state.result()                                      # the dict monthly_pipeline returned
    if not str(row.get("status", "")).startswith("BLOCKED"):
        return
    message = (f"EMR pipeline: {row['month']} was BLOCKED ({row['details']}). "
               f"The file is in data/quarantine/; nothing was scored. Please ask the EMR team for a corrected export.")
    alert_log.warning(message)
    with open(OUTBOX, "a") as f:                              # real life, for example:
        f.write(message + "\n")                              #   httpx.post(SLACK_WEBHOOK_URL, json={"text": message})


# on_failure covers the other case: a run that crashes (EMR down after all retries, a bug)
alerting_pipeline = pl.monthly_pipeline.with_options(on_completion=[notify_if_blocked],
                                                     on_failure=[lambda flow, flow_run, state: alert_log.error(
                                                         f"monthly pipeline CRASHED: {flow_run.name}")])

jan = alerting_pipeline("2025-01", "baseline")                # a normal month: no alert expected
print("January:", jan["status"], "| outbox exists:", OUTBOX.exists())
feb = alerting_pipeline("2025-02", "messy_export")            # the messy export: blocked
print("February:", feb["status"])
print("\nOutbox:\n" + OUTBOX.read_text())

**Answer.** A flow hook. `notify_if_blocked` runs when the run finishes, reads the row the pipeline returned and, only for a blocked month,
sends the message. In the output above January (normal) sent nothing (`outbox exists: False`) and the messy February produced one message with
the month, the reason and what to do next, in the log and in the outbox. In production, replace the file write with one call to your channel
(a Slack/Teams webhook, an e-mail, or a Prefect notification block). `on_failure` covers the other case, a run that crashes after all retries.
If you prefer the UI route, an Automation needs something to react to: blocked months end as `Completed`, so either make the pipeline emit an event
or tag for blocked months, or fail the run on purpose. The hook is simpler and lives in code, in git, next to the rest of the policy.
Whoever receives it should be the EMR team, not the data scientist: the fix is a corrected export.

## Exercise 2

**Human approval.** Before `set_registered_model_alias(..., "champion", ...)`, pause and ask a person: look up
`prefect.flow_runs.pause_flow_run(wait_for_input=...)`. What should the approver see?

In [ ]:
import inspect
from prefect.flow_runs import pause_flow_run, resume_flow_run
print("pause_flow_run", inspect.signature(pause_flow_run))

`wait_for_input` takes a *class* (a `RunInput`); Prefect builds a form from it in the UI and returns the filled-in object when a person
submits it. If nobody answers within `timeout`, the flow run fails and the alias is never moved: the safe default.
Two rules from the docs: `pause_flow_run` works in a **flow**, not in a task, so the alias change must move out of the
`retrain_and_decide` task into the flow body; and the person answering is a human in the UI, so the notebook plays that role with a small thread
that calls `resume_flow_run` (what the UI's "Resume" button does).

First, a candidate to approve. This uses its own temporary MLflow store, and the concept-shift months of the lesson: v1 is the 2024 model,
the challenger is trained on April to June 2025, both are compared on July and August with the pipeline's own `_benchmark` function.

In [ ]:
import mlflow
em = pl.em                                  # the helpers the pipeline uses (emr_model.py)

store = TMP / "approval_store"
store.mkdir()
mlflow.set_tracking_uri(f"sqlite:///{store / 'mlflow.db'}")
mlflow.create_experiment("approval-demo", artifact_location=(store / "artifacts").as_uri())
mlflow.set_experiment("approval-demo")
client = mlflow.MlflowClient()
NAME = "gam-risk"

shifted = TMP / "shifted"                   # concept_shift months, simulated into a temp folder
em.simulate("2025-04", 5, scenario="concept_shift", out=shifted)
def months(a, b): return em.load_months(a, b, raw=shifted)

def register(model, train_months, run_name):
    with mlflow.start_run(run_name=run_name):
        info = mlflow.sklearn.log_model(model, name="logistic-regression", pyfunc_predict_fn="predict_proba",
                                        skops_trusted_types=["numpy.dtype"])
    mv = mlflow.register_model(info.model_uri, NAME)
    client.set_model_version_tag(NAME, mv.version, "train_months", json.dumps(train_months))
    return mv

hist = pl.load([f"2024-{m:02d}" for m in range(1, 11)])
v1 = register(em.build_model("logistic").fit(hist[em.FEATURES], hist[em.LABEL]), "2024-01..2024-10", "v1")
client.set_registered_model_alias(NAME, "champion", v1.version)

train, bench = months("2025-04", "2025-06"), months("2025-07", "2025-08")
challenger = em.build_model("logistic").fit(train[em.FEATURES], train[em.LABEL])
champion = mlflow.sklearn.load_model(f"models:/{NAME}@champion")
champ, chall = pl._benchmark(champion, bench), pl._benchmark(challenger, bench)
pd.DataFrame({"champion v1": champ, "challenger": chall}).T

In [ ]:
# The pipeline's promotion rule, copied from retrain_and_decide so we can apply it here
def promotion_rule(champ, chall):
    reasons = []
    if chall["roc_auc"] < champ["roc_auc"] - 0.005:
        reasons.append("ranking worse")
    if abs(np.log(chall["obs/pred"])) >= abs(np.log(champ["obs/pred"])) - 0.05:
        reasons.append("calibration not better")
    for g in ("sens_F", "sens_M"):
        if champ[g] - chall[g] > 0.02:
            reasons.append(f"{g} drops")
    return reasons

print("rule says:", "PROMOTE" if not promotion_rule(champ, chall) else promotion_rule(champ, chall))

The rule says promote. Now the approval step. What goes in the pause message is the point of the exercise: the approver needs to decide without opening anything else.

In [ ]:
from IPython.display import Markdown, display
from prefect.context import get_run_context
from prefect.input import RunInput


class PromotionApproval(RunInput):               # becomes the form in the UI
    approved: bool
    reviewer: str
    comment: str = ""


def approval_summary(champ, chall, train_months, bench_months, flagged_note):
    row = lambda k, f="{:.3f}": f"| {k} | {f.format(champ[k])} | {f.format(chall[k])} |"
    return "\n".join([
        "## Promote the challenger to @champion?",
        f"The written rule says **PROMOTE**. Challenger trained on {train_months}; compared on {bench_months} (outcomes known).",
        "",
        "| Benchmark | Champion v1 | Challenger |", "|---|---|---|",
        row("roc_auc"), row("obs/pred"), row("sens_F"), row("sens_M"),
        "",
        "- **observed/predicted** = real GAM rate / predicted rate; 1.0 is perfectly calibrated.",
        "- **sens_F / sens_M** = share of children who developed GAM that were flagged, per sex (fairness check).",
        f"- **What changes for the clinics:** {flagged_note}",
        "- **Rollback:** point `@champion` back to version 1 in the MLflow registry (one command).",
        "",
        "Approve or reject below; add a comment when rejecting. No answer in 3 days: the model stays as it is.",
    ])

flag_note = (f"mean predicted risk goes from {champion.predict_proba(bench[em.FEATURES])[:, 1].mean():.1%} to "
             f"{challenger.predict_proba(bench[em.FEATURES])[:, 1].mean():.1%} on the same visits "
             f"(real rate {bench[em.LABEL].mean():.1%}); the top 20% of visits are still flagged, but different children.")
summary = approval_summary(champ, chall, "2025-04..2025-06", "2025-07..2025-08", flag_note)
display(Markdown(summary))

In [ ]:
seen = {}

@flow(name="promote-with-approval")
def promote_with_approval(summary_md: str, timeout: int = 3 * 24 * 3600, poll_interval: int = 1):
    seen["run_id"] = get_run_context().flow_run.id
    answer = pause_flow_run(wait_for_input=PromotionApproval.with_initial_data(description=summary_md, approved=False),
                            timeout=timeout, poll_interval=poll_interval)      # <- the run stops here until a person answers
    if answer.approved:
        mv = register(challenger, "2025-04..2025-06", "challenger")
        client.set_registered_model_alias(NAME, "champion", mv.version)         # the line the exercise is about
    return {"approved": answer.approved, "reviewer": answer.reviewer, "comment": answer.comment}


def play_approver(answer):
    """Stands in for the person: waits for the run to pause, then does what the UI's Resume button does."""
    from prefect import get_client
    while "run_id" not in seen:
        time.sleep(0.2)
    with get_client(sync_client=True) as c:
        while not c.read_flow_run(seen["run_id"]).state.is_paused():
            time.sleep(0.2)
    resume_flow_run(seen["run_id"], run_input=answer)

def run_with_answer(answer):
    seen.clear()
    approver = threading.Thread(target=play_approver, args=(answer,))
    approver.start()
    result = promote_with_approval(summary)
    approver.join()
    alias = client.get_model_version_by_alias(NAME, "champion").version
    print(f"flow returned {result}\n@champion is now version {alias}")

print("--- reviewer rejects:")
run_with_answer({"approved": False, "reviewer": "Dr. Alem", "comment": "Wait for September's outcomes first."})
print("--- reviewer approves:")
run_with_answer({"approved": True, "reviewer": "Dr. Alem", "comment": "Calibration fixed, both sexes better."})

**Answer.** The signature is `pause_flow_run(wait_for_input=None, timeout=3600, poll_interval=10, key=None)`; `wait_for_input` is a `RunInput` class
whose fields become the form. It ran here without a person: a thread did what the UI's Resume button does. The rejected run left `@champion` at
version 1; the approved run moved it to version 2. If nobody answers before `timeout`, the run fails and the alias stays put.
To use it in `pipeline.py`, move the register-and-alias lines out of the `retrain_and_decide` task into the flow body (a pause is only allowed in a flow):
the task returns the challenger and the verdict, the flow asks, and only then moves the alias.

What the approver should see, in the message above: (1) the question and the rule's verdict; (2) champion and challenger side by side on the same recent months
(ROC-AUC, observed/predicted, sensitivity for girls and boys) with a one-line meaning for each; (3) what the training and benchmark months were;
(4) what changes for the clinics (mean risk, which visits get flagged); (5) how to roll back; (6) what happens if nobody answers.
The form records who decided and a comment, so the decision is auditable. For a clinical model, keep a person in the loop; for a low-stakes one, log and promote.

## Exercise 3

**Training window.** Set `TRAIN_WINDOW = 6` and rerun (start over first). When is the challenger promoted, and how close to 1 is
observed/predicted afterwards? Why?

In [ ]:
def run_timeline(window):
    p = fresh_pipeline(f"timeline_w{window}")                  # "start over": a brand new, empty pipeline
    p.TRAIN_WINDOW = window                                    # the only change; pipeline.py is not edited
    t = time.time()
    quiet(p.timeline)
    log = pd.read_csv(p.LOG)
    print(f"TRAIN_WINDOW={window}: 15 months in {time.time() - t:.0f}s")
    return p, log

cols = ["month", "scenario (hidden truth)", "model_version", "outcomes_for", "obs/pred", "auc", "retrain"]
p6, log6 = run_timeline(6)
log6[cols].fillna("")

In [ ]:
# For comparison: the lesson's default window of 4 (README: promoted in 2025-11, observed/predicted 1.18 afterwards)
p4, log4 = run_timeline(4)

def summarise(p, log):
    promo = log[log["retrain"].fillna("").str.startswith("PROMOTE")]
    month = promo["month"].iloc[0]
    version = p._mlflow().MlflowClient().get_model_version_by_alias(p.MODEL_NAME, "champion")
    trained = json.loads(version.tags["train_months"])
    after = log[log["outcomes_for"].fillna("").str.contains("scored by v2")]
    post_shift = [m for m in trained if m >= "2025-04"]
    return {"promoted in": month, "challenger trained on": f"{trained[0]}..{trained[-1]} ({len(trained)} months)",
            "of which after the shift (Apr+)": f"{len(post_shift)} of {len(trained)}",
            "first obs/pred of a v2-scored month": float(after["obs/pred"].iloc[0]),
            "that month": after["outcomes_for"].iloc[0]}

pd.DataFrame({"TRAIN_WINDOW=4": summarise(p4, log4), "TRAIN_WINDOW=6": summarise(p6, log6)})

In [ ]:
# Why not earlier? Every challenger the pipeline trained, from MLflow: how well calibrated vs the champion, on its benchmark
def challengers(p):
    runs = p._mlflow().search_runs(experiment_names=["gam-risk-pipeline"], order_by=["start_time ASC"])
    runs = runs[runs["tags.mlflow.runName"].fillna("").str.startswith("challenger-")].copy()
    runs["trained in month"] = runs["tags.mlflow.runName"].str.removeprefix("challenger-")
    runs["post-shift months in training"] = runs["params.train_months"].apply(
        lambda s: f"{sum(m >= '2025-04' for m in s.split(','))} of {len(s.split(','))}")
    return runs.set_index("trained in month")[["post-shift months in training", "metrics.champion_obs_pred",
                                               "metrics.challenger_obs_pred", "metrics.champion_roc_auc", "metrics.challenger_roc_auc"]]

pd.concat({"TRAIN_WINDOW=4": challengers(p4), "TRAIN_WINDOW=6": challengers(p6)}).rename(columns=lambda c: c.replace("metrics.", ""))

**Answer.** With `TRAIN_WINDOW = 6` the challenger is promoted in **2025-11, the same month as with the default window of 4**. The tables show why the window does
not change when: a challenger only beats the champion's calibration once **at least 3 of its training months are after the April shift**
(0, 0, 1 and 2 post-shift months in July to October: observed/predicted stays about 1.5 to 2.2; with 3 in November it drops to 0.98 in both runs). That depends on the calendar, not on the window size.

It does change how well the new model fits afterwards. The first month scored by v2 with known outcomes (December) has observed/predicted **1.46 with window 6, against 1.18 with window 4**:
better than the 1.9 to 2.2 before, but further from 1. Reason: a logistic regression learns one average relationship, and the 6-month window holds only 3 post-shift months out of 6 (December, January and
March are from before the shift), while the 4-month window holds 3 out of 4. The old months pull the model back toward the old world. A longer window gives more stable estimates, a shorter one adapts faster;
after a shift, you want to forget quickly. (Also note the benchmark months, July and August, are lean season and the check month, December, is not, so the
two are not like for like. One month is thin evidence.)

## Exercise 4

**Fix and re-run February.** Clean the quarantined file (GX lesson exercise 4), put it back in `data/incoming/`, and run
`python pipeline.py month 2025-02 messy_export`. What does the monitoring log show?

In [ ]:
# In the base pipeline February was blocked and the file quarantined (exercise 1)
bad_path = pl.QUARANTINE / "emr_2025-02.csv"
print("quarantined:", [p.name for p in pl.QUARANTINE.glob("*.csv")])
messy = pd.read_csv(bad_path)
print(len(messy), "rows;", messy["visit_id"].duplicated().sum(), "duplicate visit_ids")

In [ ]:
def clean_export(df):
    """The GX lesson's idea: fix the DATA, not the rules. Returns the clean frame and what each step did."""
    steps = {}
    n = len(df)
    df = df.drop_duplicates().reset_index(drop=True)                  # re-run of part of the export
    steps["exact duplicate rows dropped"] = n - len(df)
    df["sex"] = df["sex"].replace({"female": "F", "male": "M"})        # free-text codes from one clerk
    steps["sex codes mapped"] = int((~messy["sex"].isin(["F", "M"])).sum())
    mm = df["muac_cm"] > 50                                            # Amhara sent millimetres
    df.loc[mm, "muac_cm"] = df.loc[mm, "muac_cm"] / 10
    steps["MUAC converted mm -> cm"] = int(mm.sum())
    infants = (df["age_months"] < 6) & df["muac_cm"].notna()            # protocol: no MUAC under 6 months
    df.loc[infants, "muac_cm"] = np.nan
    steps["infant MUAC blanked"] = int(infants.sum())
    n = len(df)
    df = df[(df["age_months"] <= 59) & (df["whz"].isna() | (df["whz"] >= -2))]   # aged out / should have been referred
    steps["out-of-cohort rows removed"] = n - len(df)
    return df.reset_index(drop=True), steps

clean, steps = clean_export(messy)
print(json.dumps(steps, indent=1), f"\n{len(messy)} rows -> {len(clean)} rows")

# Put the cleaned file back in data/incoming/ and set the quarantined original aside (kept as evidence)
(pl.QUARANTINE / "resolved").mkdir(exist_ok=True)
shutil.move(bad_path, pl.QUARANTINE / "resolved" / "emr_2025-02.original.csv")
clean.to_csv(pl.INCOMING / "emr_2025-02.csv", index=False)

In [ ]:
# Same call as `python pipeline.py month 2025-02 messy_export`. The ingest step finds the file already in
# data/incoming/ and leaves it alone, so the cleaned file is what gets validated.
row = pl.monthly_pipeline("2025-02", "messy_export")
print(json.dumps(row, indent=1, default=str))

In [ ]:
pd.read_csv(pl.LOG)[["month", "status", "details", "model_version", "mean_risk", "drift_gate", "outcomes_for"]].fillna("")

**Answer.** The log keeps both rows for 2025-02: first **BLOCKED by GX** (5 critical checks failed, nothing scored), then, after the cleaned file went through,
**ok**: scored by v1, mean risk 0.089 (January: 0.092), drift gate `ok`, a new data version hash. The log is append-only on purpose, so the history of what happened stays visible; the original messy file is kept in
`data/quarantine/resolved/` as evidence.

The cleaning, step by step (2015 rows in, 1963 out): 15 duplicate rows dropped, 41 free-text sex codes mapped, 416 MUAC values converted from mm to cm,
78 infant MUAC values blanked (protocol: no MUAC under 6 months, which GX flagged and Amhara's new EMR version had filled), and 37 out-of-cohort rows removed
(children aged 60+ months or with WHZ below -2, who should have been referred). Whether this belongs in the pipeline: only as a stop-gap. The EMR export should be fixed upstream; a cleaning step hides
the problem, and silently deleting rows changes who gets scored, so the counts above should be logged each time it runs.

## Exercise 5

**An edge bundle.** Add a final task that exports the champion as a points table plus input rules for offline devices
(see the main README). When should it run?

The champion is a logistic regression, so its log-odds is a sum of one contribution per input. Dividing each contribution by a
"point" (0.25 log-odds) and rounding gives a **points table** a phone or paper form can add up, plus a short table to turn the total into a risk.
Continuous inputs are cut into 5 bands. The **input rules** tell the device what values are acceptable, so a value in the wrong unit
(MUAC in mm) is rejected on the device instead of silently scored.

In [ ]:
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

POINT = 0.25                                           # log-odds per point
CONTINUOUS = ["age_months", "weight_kg", "height_cm", "whz", "muac_cm", "mother_age", "water_source_minutes", "household_size"]
PLAUSIBLE = {"age_months": (0, 59), "weight_kg": (1.5, 30), "height_cm": (40, 125), "whz": (-5, 5), "muac_cm": (5, 25),
             "mother_age": (15, 49), "water_source_minutes": (0, 600), "household_size": (1, 30)}   # to be reviewed by clinicians


def points_table(model, train, point=POINT, bands=5):
    """Fitted sklearn pipeline -> (points DataFrame, base log-odds for 0 points)."""
    pre, clf = model.steps[0][1], model.steps[-1][1]
    imp, scaler = pre.named_transformers_["num"].steps
    imp, scaler, enc = imp[1], scaler[1], pre.named_transformers_["cat"]
    coef = clf.coef_[0]
    n_num = len(scaler.mean_)                                          # numeric inputs + missing-value flags
    per_unit = coef[:n_num] / scaler.scale_                            # log-odds per unit of each numeric input
    median = dict(zip(em.NUMERIC, imp.statistics_))                    # a missing value is replaced by the median
    reference = np.r_[[median[c] for c in em.NUMERIC], np.zeros(n_num - len(em.NUMERIC))]
    base = clf.intercept_[0] + float(np.sum(coef[:n_num] * (reference - scaler.mean_) / scaler.scale_))
    pts = lambda logit: int(round(logit / point))
    rows = []
    for j, c in enumerate(em.NUMERIC):
        if c in CONTINUOUS:
            x = train[c].dropna()
            edges = np.unique(np.quantile(x, np.linspace(0, 1, bands + 1)))
            edges[0], edges[-1] = PLAUSIBLE[c]
            for lo, hi in zip(edges[:-1], edges[1:]):
                mid = float(x[(x >= lo) & (x <= hi)].median())
                rows.append((c, f"{lo:g} to {hi:g}", lo, hi, pts(per_unit[j] * (mid - median[c]))))
        else:                                                           # 0/1 inputs
            rows += [(c, str(v), v, v, pts(per_unit[j] * (v - median[c]))) for v in (0, 1)]
    for k, i in enumerate(imp.indicator_.features_):                    # "value missing" flags the model learned
        rows.append((em.NUMERIC[i], "missing", None, None, pts(per_unit[len(em.NUMERIC) + k])))
    offset = n_num
    for c, cats in zip(em.CATEGORICAL, enc.categories_):                # categories: points relative to the lowest one
        cc = coef[offset:offset + len(cats)]
        offset += len(cats)
        base += cc.min()
        rows += [(c, str(cat), None, None, pts(v - cc.min())) for cat, v in zip(cats, cc)]
    return pd.DataFrame(rows, columns=["input", "value", "lo", "hi", "points"]), float(base)


def total_points(df, table):
    """What the device does: look up each input in the table and add the points."""
    total = np.zeros(len(df), dtype=int)
    for c, t in table.groupby("input", sort=False):
        miss = t.loc[t["value"] == "missing", "points"]
        rest = t[t["value"] != "missing"]
        if c in CONTINUOUS:
            edges = np.r_[rest["lo"].iloc[0], rest["hi"].values]
            idx = np.clip(np.searchsorted(edges, df[c].fillna(edges[0]).values, side="right") - 1, 0, len(rest) - 1)
            p = rest["points"].values[idx]
        elif c in em.CATEGORICAL:
            p = df[c].astype(str).map(dict(zip(rest["value"], rest["points"]))).fillna(0).values
        else:
            p = df[c].map({0: rest.loc[rest["value"] == "0", "points"].iloc[0],
                           1: rest.loc[rest["value"] == "1", "points"].iloc[0]}).fillna(0).values
        total += np.where(df[c].isna(), miss.iloc[0] if len(miss) else 0, p).astype(int)
    return total


def input_rules(model, version):
    enc = model.steps[0][1].named_transformers_["cat"]
    fields = {c: {"type": "number", "min": lo, "max": hi, "missing_allowed": c in ("muac_cm", "weight_kg", "height_cm", "whz")}
              for c, (lo, hi) in PLAUSIBLE.items()}
    fields["muac_cm"]["note"] = "centimetres! values above 25 are probably millimetres: reject"
    fields |= {c: {"type": "category", "allowed": [str(v) for v in cats], "missing_allowed": False}
               for c, cats in zip(em.CATEGORICAL, enc.categories_)}
    fields |= {c: {"type": "number", "allowed": [0, 1], "missing_allowed": False}
               for c in ["recent_diarrhea", "recent_fever", "vaccinated_for_age", "currently_breastfed",
                         "prior_sam_admission", "lean_season"]}
    return {"model": f"gam-risk v{version}", "fields": fields,
            "cohort": ["age_months must be 0-59", "whz >= -2 and muac_cm >= 12.5 (already malnourished: refer, do not score)"]}

In [ ]:
@task
def export_edge_bundle(p, out_dir):
    """Final task: export the current @champion as a points table + input rules + risk lookup, as plain files."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    mlflow = p._mlflow()
    mv = mlflow.MlflowClient().get_model_version_by_alias(p.MODEL_NAME, "champion")
    model = mlflow.sklearn.load_model(f"models:/{p.MODEL_NAME}@champion")
    train = p.load(json.loads(mv.tags["train_months"]))                 # the data the champion learned from
    table, base = points_table(model, train)
    scores = total_points(train, table)
    risk_by_points = {int(s): round(float(1 / (1 + np.exp(-(base + POINT * s)))), 4) for s in range(scores.min(), scores.max() + 1)}
    bundle = {"model_version": int(mv.version), "point_value_log_odds": POINT,
              "flag_at_or_above_points": int(np.quantile(scores, 0.8)),       # top 20%, the clinics' capacity
              "risk_by_total_points": risk_by_points}
    table.to_csv(out / f"points_table_v{mv.version}.csv", index=False)
    (out / f"input_rules_v{mv.version}.json").write_text(json.dumps(input_rules(model, mv.version), indent=1))
    (out / f"risk_lookup_v{mv.version}.json").write_text(json.dumps(bundle, indent=1))
    return {"version": int(mv.version), "folder": str(out), "table": table, "base": base, "bundle": bundle, "model": model}


@flow(name="after-promotion")
def after_promotion(p, out_dir):
    return export_edge_bundle(p, out_dir)                                # more tasks could follow: publish, notify devices team

edge = after_promotion(pl, TMP / "edge_v1")
print("files:", sorted(f.name for f in Path(edge["folder"]).iterdir()))
edge["table"].drop(columns=["lo", "hi"]).groupby("input", sort=False).agg(values=("value", "count"),
    lowest=("points", "min"), highest=("points", "max")).T

In [ ]:
# The first rows of the points table, and does adding points rank children like the model does?
display_rows = edge["table"][edge["table"]["input"].isin(["whz", "muac_cm", "recent_diarrhea", "region"])]
print(display_rows.drop(columns=["lo", "hi"]).to_string(index=False))

test = pl.load(["2024-11", "2024-12"])                                    # months the v1 model did not train on
model_risk = edge["model"].predict_proba(test[em.FEATURES])[:, 1]
pts_ = total_points(test, edge["table"])
est = 1 / (1 + np.exp(-(edge["base"] + POINT * pts_)))
top = lambda s: s >= np.quantile(s, 0.8)
print(f"\nROC-AUC  model {roc_auc_score(test[em.LABEL], model_risk):.3f}   points {roc_auc_score(test[em.LABEL], pts_):.3f}")
print(f"rank correlation (Spearman) {spearmanr(model_risk, pts_)[0]:.3f};  same children flagged (top 20%): "
      f"{(top(model_risk) & top(pts_)).sum() / top(model_risk).sum():.0%}")
print(f"mean risk  model {model_risk.mean():.3f}   from points {est.mean():.3f}")

In [ ]:
# What the device checks before scoring: one good record, one with MUAC in millimetres
rules = json.loads((Path(edge["folder"]) / f"input_rules_v{edge['version']}.json").read_text())["fields"]

def problems(record):
    out = []
    for name, rule in rules.items():
        v = record.get(name)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            if not rule["missing_allowed"]:
                out.append(f"{name}: required")
        elif "allowed" in rule and (str(v) not in [str(a) for a in rule["allowed"]]):
            out.append(f"{name}: {v!r} not allowed")
        elif "min" in rule and not (rule["min"] <= v <= rule["max"]):
            out.append(f"{name}: {v} outside {rule['min']}-{rule['max']}")
    return out

child = test.iloc[0][em.FEATURES].to_dict()
print("good record:", problems(child) or "ok", "| points:", int(total_points(test.iloc[[0]], edge["table"])[0]))
print("muac in mm: ", problems({**child, "muac_cm": 135.0}))

In [ ]:
# The bundle after a promotion: the timeline with TRAIN_WINDOW=6 promoted v2, so export again from that pipeline
edge2 = after_promotion(p6, TMP / "edge_v2")
t1, t2 = edge["table"].set_index(["input", "value"])["points"], edge2["table"].set_index(["input", "value"])["points"]
both = pd.concat({"v1": t1, f"v{edge2['version']} (after promotion)": t2}, axis=1, join="inner")   # same keys only
both = both[~both.index.get_level_values("input").isin(CONTINUOUS)]       # bands differ between versions: compare 0/1 and category inputs
changed = both[(both.iloc[:, 0] != both.iloc[:, 1])]
print(f"files for v{edge2['version']}:", sorted(f.name for f in Path(edge2['folder']).iterdir()))
print(f"{len(changed)} of {len(both)} yes/no and category entries changed; the largest changes:")
changed.assign(change=(changed.iloc[:, 1] - changed.iloc[:, 0]).abs()).sort_values("change", ascending=False).head(6).drop(columns="change")

**Answer.** `export_edge_bundle` loads `@champion`, reads the months it was trained on from its registry tags, and writes three files: a **points table** (every input band or category with
its points; the sum of points is the score), **input rules** (allowed ranges and categories, which fields may be missing; a MUAC of 135, an mm value, is rejected) and a **risk lookup**
(total points to risk, and the point total at which to flag the top 20%). It runs in a small `after-promotion` flow.
Fidelity check on two months the model did not train on: ROC-AUC 0.805 from points against 0.812 from the model, 0.975 rank correlation, and 95% of the same children flagged.
The risk read from the points table is a bit high (mean 0.099 against 0.089), so use the points to rank and re-check the calibration before showing percentages.

**When it should run:** right after a promotion, that is when `@champion` moves (and once after bootstrap for the first model), after the human approval of exercise 2 if there is one,
not every month: the files only change when the model changes. The v2 bundle shows why it matters: after the concept shift,
`recent_diarrhea` is worth 4 points instead of 2 and the lean season 5 instead of 2. Devices still carrying v1 would keep under-weighting illness, so the devices team must be told (and the old bundle kept for rollback).

---
Closing the temporary Prefect server (the next cell). Everything under the temporary folder can be deleted.

In [ ]:
harness.__exit__(None, None, None)
shutil.rmtree(TMP, ignore_errors=True)
print("temporary Prefect server stopped; temporary folder removed")